# Compute per-site band quality baseline
* Downloads (if not already present) the 30 min low-tide, 0% cloud satellite images per site across 2016-2026
* Computes the per-site, per-band median/MAD baseline (see sentinel2.compute_and_save_band_quality_baseline)
* Only needs to be run once - the baseline is cached to utils.get_band_quality_baseline_path() and reused by sentinel2.flag_anomalous_band_dates

In [1]:
import geopandas
import pathlib
import pandas
import dask.distributed

module_path = pathlib.Path.cwd().parent / 'scripts'
import sys
if str(module_path) not in sys.path:
    sys.path.append(str(module_path))
import utils
import sentinel2

%load_ext autoreload
%autoreload 2

In [2]:
cluster = dask.distributed.LocalCluster()
client = dask.distributed.Client(cluster)
display(client)

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 8,Total memory: 31.73 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:55541,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:53177,Total threads: 2
Dashboard: http://127.0.0.1:53178/status,Memory: 7.93 GiB
Nanny: tcp://127.0.0.1:55544,


# Values to edit

In [3]:
all_training_sites =  ["Ihutai", "CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua", "Purau",
              "IveyBay_Nov25", "IveyBay_Feb26", "LeftBank_Nov25", "LeftBank_Feb26", "Paremata_Nov25", "Paremata_Feb26",
              "Paremata_Feb25", "ThePoint_Nov25", "ThePoint_Feb26", "Takapuwahia_Nov25", "Takapuwahia_Feb26", "IveyBay_ThePoint_LeftBank_Oct24"]

In [4]:
data_path = utils.get_data_path()
utils.create_data_folders()

In [5]:
sites = all_training_sites
mndwi_file = data_path / "mndwi_low_tide_thresholds.csv"
baseline_max_cloud_cover = 0
years = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

# Cells to run
* Download the low-tide no-cloud images by year and site (skips sites already downloaded)
* Compute and save the per-site, per-band quality baseline

### Download satellite images
The 0% cloud, low-tide images across sites for each year and tide delta (30min/45min/1hr),
reusing the prediction-style per-year storage. Pooling across tide deltas (rather than dropping
the low-tide requirement entirely) grows the per-site sample size while keeping the baseline
restricted to genuinely low-tide reflectance.

In [6]:
_, satellite_path = utils.get_prediction_path(
    sample_method="sampling_2", method_2_threshold=0.5,
    model_max_cloud_cover=baseline_max_cloud_cover,
    predict_max_cloud_cover=baseline_max_cloud_cover,
    use_threshold=True, satellite_only=True) 

In [7]:
mndwi_thresholds = pandas.read_csv(mndwi_file)

In [8]:
mndwi_thresholds

,site,threshold
0,Ihutai,0.285990
1,CatlinsLake,0.324971
2,CatlinsRiverMouth,0.077602
3,Childrens,0.351367
4,Duvauchelle,0.378407
5,Robinsons,0.321775
6,Takamatua,0.455245
7,Purau,0.472248
8,IveyBay_Nov25,0.202730
9,IveyBay_Feb26,0.202730


In [14]:
for site in sites:
    mndwi_threshold = mndwi_thresholds.loc[mndwi_thresholds["site"] == site, "threshold",].squeeze()
    if not isinstance(mndwi_threshold, float):
        print(f"WARNING: Expected a float tide threshold for {site}. Exiting.")
        break

    # Skip if already run
    if (satellite_path / f"{site}_{years[-1]}_sentinel-2.nc").exists():
        print(f"\tAlready run for {site} as {site}_{years[-1]}_sentinel-2.nc exists. "
              "Delete if you want redownloaded.")
        continue
    
    for year in years:
        print(f"{site}: year {year}")

        satellite_file = satellite_path / f"{site}_{year}_sentinel-2.nc"
        site_polygon_file = utils.get_site_polygon_path(site)

        if not satellite_file.exists():
            print(f"\tSave satellite image of the site {site} MNDWI thresholded and cloud filtered in the year {year}")
            site_polygon = geopandas.read_file(site_polygon_file)
            for i in range(3):
                try:
                    satellite_data = sentinel2.get_mndwi_and_cloud_filtered_images_in_year(
                        geometry=site_polygon,
                        max_cloud_cover=baseline_max_cloud_cover,
                        year=year,
                        mndwi_threshold=mndwi_threshold,
                    )
                    break
                except Exception:
                    print("Caught exception downloading images and try again.")

            if len(satellite_data['time']) == 0:
                print("Warning: No satellite data without cloud. Ignore")
            else:
                print("\t\tSave")
                utils.write_netcdf_conventions_in_place(satellite_data)
                satellite_data = satellite_data.rio.reproject(utils.CRS_NZTM)
                satellite_data = satellite_data.rio.clip(site_polygon.geometry, drop=True, all_touched=True)
                utils.write_netcdf_conventions_in_place(satellite_data)
                utils.save_netcdf(satellite_data, satellite_file)

	Already run for Ihutai as Ihutai_2026_sentinel-2.nc exists. Delete if you want redownloaded.
	Already run for CatlinsLake as CatlinsLake_2026_sentinel-2.nc exists. Delete if you want redownloaded.
	Already run for CatlinsRiverMouth as CatlinsRiverMouth_2026_sentinel-2.nc exists. Delete if you want redownloaded.
	Already run for Childrens as Childrens_2026_sentinel-2.nc exists. Delete if you want redownloaded.
	Already run for Duvauchelle as Duvauchelle_2026_sentinel-2.nc exists. Delete if you want redownloaded.
	Already run for Robinsons as Robinsons_2026_sentinel-2.nc exists. Delete if you want redownloaded.
	Already run for Takamatua as Takamatua_2026_sentinel-2.nc exists. Delete if you want redownloaded.
	Already run for Purau as Purau_2026_sentinel-2.nc exists. Delete if you want redownloaded.
	Already run for IveyBay_Nov25 as IveyBay_Nov25_2026_sentinel-2.nc exists. Delete if you want redownloaded.
	Already run for IveyBay_Feb26 as IveyBay_Feb26_2026_sentinel-2.nc exists. Delete 

### Compute and save the band quality baseline
Pools all downloaded years across all tide deltas per site into one dataset, then computes
per-site, per-band median/MAD and caches it to utils.get_band_quality_baseline_path().

In [8]:
baseline = sentinel2.compute_and_save_band_quality_baseline(
    training_sites=sites,
    satellite_images_paths=[satellite_path],
)
baseline

	Saved band quality baseline for 20 sites to C:\Local\repos\seagrass-detection\data\band_quality_baseline.csv


,site,band,median,mad,mad_raw,n_images
0,Ihutai,B04,564.00,98.823986,46.00,7
1,Ihutai,B03,668.00,100.625750,34.00,7
2,Ihutai,B02,570.00,94.723457,38.00,7
3,CatlinsLake,B04,451.00,72.215061,33.00,23
4,CatlinsLake,B03,424.00,81.718494,45.00,23
5,CatlinsLake,B02,315.00,72.012403,35.00,23
6,CatlinsRiverMouth,B04,849.00,94.680177,75.00,23
7,CatlinsRiverMouth,B03,932.00,102.578959,84.00,23
8,CatlinsRiverMouth,B02,637.00,87.524031,64.00,23
9,Childrens,B04,432.50,110.165670,108.00,66


In [10]:
import pandas
pandas.read_csv(r"C:\Local\repos\seagrass-detection\data\band_quality_baseline_tide_api.csv")

,site,band,median,mad,mad_raw,n_images
0,Ihutai,B04,502.00,51.494618,8.00,16
1,Ihutai,B03,642.00,59.765278,31.00,16
2,Ihutai,B02,568.00,53.913715,29.00,16
3,CatlinsLake,B04,356.00,57.154375,45.50,50
4,CatlinsLake,B03,378.00,63.650714,56.00,50
5,CatlinsLake,B02,262.00,52.527054,44.00,50
6,CatlinsRiverMouth,B04,814.00,93.410750,96.00,55
7,CatlinsRiverMouth,B03,877.00,85.140667,86.00,55
8,CatlinsRiverMouth,B02,637.00,50.491917,42.00,55
9,Childrens,B04,423.00,82.479365,82.00,159
